# Salvar

A reta treinada vira uma pasta. Essa pasta é o formato de deploy do MLflow.


In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import pandas as pd
import mlflow
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("cerveja")

dataset = pd.read_csv("dados/Consumo_cerveja.csv")
X = dataset[["Temperatura Maxima (C)"]].to_numpy()
y = dataset["Consumo de cerveja (litros)"].to_numpy()
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, random_state=0)
modelo = LinearRegression().fit(X_treino, y_treino)


/Users/teixeirah/work/uri/Ciência de Dados/aula-ml-2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# A pasta modelo/ contém o arquivo MLmodel. É ela que o deploy publica.
# Se já existir, apaga para poder gravar de novo.
import shutil
shutil.rmtree("modelo", ignore_errors=True)
mlflow.sklearn.save_model(modelo, "modelo")


2026/10/08 19:01:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [3]:
# Lê a pasta de volta. 30 é a temperatura máxima do dia, em °C.
carregado = mlflow.sklearn.load_model("modelo")
carregado.predict([[30]])


array([27.5029209])

In [4]:
# A mesma pasta entra na run, na coluna Models.
with mlflow.start_run(run_name="modelo-salvo"):
    mlflow.sklearn.log_model(modelo, name="modelo")


2026/10/08 19:01:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Na interface, a run `modelo-salvo` mostra o modelo na coluna Models.

`MLFLOW_ALLOW_FILE_STORE=true mlflow ui --backend-store-uri mlruns`
